## 0 · 加载 + 工具

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 自动找 data 文件夹
CANDS = [
    '../../workshop_build/workshop_build/09_advanced_ml/gan/data',
    'workshop_build/workshop_build/09_advanced_ml/gan/data',
    'data', '../data', '../../gan/data',
]
DATA = None
for guess in CANDS:
    if os.path.exists(os.path.join(guess, 'stakeholder_relations.parquet')):
        DATA = guess; break
print('文件夹：', DATA)

df = pd.read_parquet(os.path.join(DATA, 'stakeholder_relations.parquet'))
print('数据形状：', df.shape, '(列 = 街区, 栏 = 特征)')
df.head()

In [ ]:
# 让 matplotlib 显示中文（找不到字体就略过，不影响程序）
from matplotlib import font_manager as fm
for fp in [
    '../../.venv/Lib/site-packages/matplotlib/mpl-data/fonts/ttf/NotoSansCJKsc-Regular.otf',
    '../../../.venv/Lib/site-packages/matplotlib/mpl-data/fonts/ttf/NotoSansCJKsc-Regular.otf',
    'C:/Windows/Fonts/msjh.ttc',
]:
    if os.path.exists(fp):
        try:
            fm.fontManager.addfont(fp)
            plt.rcParams['font.sans-serif'] = [fm.FontProperties(fname=fp).get_name()]
            break
        except Exception: pass
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 选特征栏 + 标准化（ML 三本都要的前处理）
drop = ['site', 'gx', 'gy']
feat_cols = [c for c in df.columns
             if c not in drop and np.issubdtype(df[c].dtype, np.number)]
X = df[feat_cols].to_numpy(dtype='float32')
mu = X.mean(axis=0); sd = X.std(axis=0); sd[sd < 1e-8] = 1.0
Xs = (X - mu) / sd

share_cols = ['share_state','share_developer','share_resident','share_unknown']
share_idx  = [feat_cols.index(c) for c in share_cols]
names  = ['state 政府','developer 开发商','resident 居民','unknown 未知']
colors = ['#1f77b4','#d62728','#2ca02c','#7f7f7f']
dominant = X[:, share_idx].argmax(axis=1)
print('特征栏 %d 个，标准化后 Xs =' % len(feat_cols), Xs.shape)

## 1 · 直觉：什么叫「降维」？

想像每个街区有 13 个数字 → 它是 **13 维空间里的一个点**，看不见。
降维 = 把它**投影**到 2 维纸面，让我们**看得到**、又尽量不失真。

下图：把 3 维数据（左）压成 2 维（右）的示意。

In [ ]:
rng = np.random.default_rng(0)
n = 300
t = rng.normal(size=n)
toy3 = np.stack([t, 0.5*t + 0.3*rng.normal(size=n), -0.8*t + 0.3*rng.normal(size=n)], axis=1)

fig = plt.figure(figsize=(11,4))
ax = fig.add_subplot(1,2,1, projection='3d')
ax.scatter(toy3[:,0], toy3[:,1], toy3[:,2], s=8, c=t, cmap='viridis')
ax.set_title('原始：3 维（难看懂）')
from sklearn.decomposition import PCA
toy2 = PCA(n_components=2).fit_transform(toy3)
ax2 = fig.add_subplot(1,2,2)
ax2.scatter(toy2[:,0], toy2[:,1], s=8, c=t, cmap='viridis')
ax2.set_title('PCA 压成 2 维（看得懂了）'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2 · PCA：找「最分散」的方向

PCA 做的事：在数据里找一条**让点散得最开**的直线（PC1），再找第二条垂直的（PC2），
把每个点投影到这两条在线 → 得到 2 个座标。**只能用直线，不会弯。**

下图用玩具 2 维数据，画出 PCA 找到的两个主方向（箭头）。

In [ ]:
from sklearn.decomposition import PCA
pts = rng.normal(size=(400,2)) @ np.array([[2.0,0.8],[0.0,0.6]])
p = PCA(n_components=2).fit(pts)
center = pts.mean(0)

plt.figure(figsize=(6,6))
plt.scatter(pts[:,0], pts[:,1], s=10, alpha=0.4, color='#888')
for k in range(2):
    vec = p.components_[k] * np.sqrt(p.explained_variance_[k]) * 2.5
    plt.arrow(center[0], center[1], vec[0], vec[1], width=0.05,
              color=['#d62728','#1f77b4'][k], length_includes_head=True,
              label=f'PC{k+1}（抓 {p.explained_variance_ratio_[k]*100:.0f}% 信息）')
plt.legend(); plt.axis('equal'); plt.grid(alpha=0.3)
plt.title('PCA：红=最分散方向 PC1，蓝=次方向 PC2'); plt.show()

**重点**：PC1 抓最多信息，PC2 次之。`explained_variance_ratio_` 告诉你每个方向抓住多少 %。


In [ ]:
# 真实数据上的 PCA（13 维  2 维）
pca = PCA(n_components=2); Zp = pca.fit_transform(Xs)
plt.figure(figsize=(7,5.5))
for i in range(4):
    m = (dominant == i)
    plt.scatter(Zp[m,0], Zp[m,1], s=8, c=colors[i], label=names[i], alpha=0.6)
plt.title(f'真实街区 PCA（2 方向共抓 {pca.explained_variance_ratio_.sum()*100:.0f}% 信息）')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 3 · AE 自编码器：让网络自己学压法

PCA 只能直线。**AE** 用神经网络：中间掐一个「**瓶颈**」只有 2 个神经元，
强迫它把 13 个数字挤进 2 个、再还原回 13 个。为了还原得准，那 2 个数字就学成最有用的压缩。

```
13 维 →[encoder]→ 2 维(瓶颈 latent) →[decoder]→ 13 维
        压缩                              还原
```
因为中间有 ReLU 会弯，**AE 的 2 维图通常比 PCA 更会把不同群分开**。

In [ ]:
import matplotlib.patches as mp
layers = [13,16,2,16,13]; labels=['输入\n13','','瓶颈\n2','','输出\n13']
plt.figure(figsize=(9,4)); ax=plt.gca()
for li,(nser) in enumerate(layers):
    xs = li*2.2
    ys = np.linspace(-nser/2, nser/2, nser)
    col = '#d62728' if nser==2 else '#4C72B0'
    ax.scatter([xs]*nser, ys, s=120, color=col, zorder=3)
    if labels[li]: ax.text(xs, max(ys)+1.2, labels[li], ha='center', fontsize=11)
ax.text(2.2,-9,'encoder（压缩）',ha='center',color='#555')
ax.text(6.6,-9,'decoder（还原）',ha='center',color='#555')
ax.set_title('AE 结构：中间瓶颈只有 2 个 → 被迫学会精华压缩')
ax.axis('off'); ax.set_ylim(-10,9); plt.show()

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0); D=Xs.shape[1]
class AE(nn.Module):
    def __init__(s):
        super().__init__()
        s.enc=nn.Sequential(nn.Linear(D,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,2))
        s.dec=nn.Sequential(nn.Linear(2,16),nn.ReLU(),nn.Linear(16,32),nn.ReLU(),nn.Linear(32,D))
    def forward(s,x): z=s.enc(x); return s.dec(z), z
ae=AE(); xt=torch.tensor(Xs); opt=torch.optim.Adam(ae.parameters(),1e-3); lf=nn.MSELoss()
for e in range(60):
    xh,z=ae(xt); l=lf(xh,xt); opt.zero_grad(); l.backward(); opt.step()
print('AE 还原误差：', round(l.item(),4))
ae.eval()
with torch.no_grad(): _,Za=ae(xt)
Za=Za.numpy()
plt.figure(figsize=(7,5.5))
for i in range(4):
    m=(dominant==i); plt.scatter(Za[m,0],Za[m,1],s=8,c=colors[i],label=names[i],alpha=0.6)
plt.title('AE 的 2 维 latent（会弯曲，聚类常比 PCA 清楚）')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 4 · VAE 变分自编码器：会「生成」的 AE

AE 的 latent 有很多「洞」——你随便指一个座标 decode，可能得到垃圾。
**VAE** 多做两件事，把 latent 整理成一团干净的云（中心在原点）：
1. encoder 不输出一个点，而是输出一朵**小云**（中心 `mu`、大小 `logvar`），从云里**抽一点**。
2. 加一个 **KL** 惩罚，逼所有云都靠近原点、长得规矩。

整理干净后 → **随便从原点附近抽座标 decode，就能生成「像真的但全新」的数据**。

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(10,4.6))
a=rng.normal(size=(200,2))*np.array([3,0.4]); a=a@np.array([[1,1],[0,1]])+np.array([4,4])
ax[0].scatter(a[:,0],a[:,1],s=10,color='#4C72B0'); ax[0].set_title('AE latent：挤一边、有洞\n随便抽点 decode  可能是垃圾')
ax[0].scatter([0],[0],marker='*',s=200,color='red'); ax[0].annotate('想生成?\n这里是洞',(0,0),(-3,-3),color='red')
v=rng.normal(size=(200,2)); ax[1].scatter(v[:,0],v[:,1],s=10,color='#55A868')
ax[1].scatter([0.5],[0.4],marker='*',s=200,color='red'); ax[1].annotate('抽这里\ndecode新数据',(0.5,0.4),(1.4,1.6),color='red')
ax[1].set_title('VAE latent：被 KL 整成原点附近的云\n随便抽  生成新数据')
for a_ in ax: a_.grid(alpha=0.3); a_.axis('equal')
plt.tight_layout(); plt.show()

In [ ]:
import torch.nn.functional as F
torch.manual_seed(0)
class VAE(nn.Module):
    def __init__(s):
        super().__init__()
        s.body=nn.Sequential(nn.Linear(D,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU())
        s.mu=nn.Linear(16,2); s.lv=nn.Linear(16,2)
        s.dec=nn.Sequential(nn.Linear(2,16),nn.ReLU(),nn.Linear(16,32),nn.ReLU(),nn.Linear(32,D))
    def enc(s,x): h=s.body(x); return s.mu(h), s.lv(h)
    def forward(s,x):
        m,lv=s.enc(x); z=m+torch.exp(0.5*lv)*torch.randn_like(lv); return s.dec(z),m,lv
v=VAE(); opt=torch.optim.Adam(v.parameters(),1e-3)
for e in range(120):
    xh,m,lv=v(xt); rec=F.mse_loss(xh,xt)
    kl=(-0.5*(1+lv-m**2-lv.exp()).sum(1)).mean(); loss=rec+1.0*kl
    opt.zero_grad(); loss.backward(); opt.step()
print('VAE 还原=%.3f  KL=%.3f'%(rec.item(),kl.item()))

# 生成：从原点附近随机抽 800 点 decode
v.eval()
with torch.no_grad():
    gen=v.dec(torch.randn(800,2)).numpy()
real2=PCA(n_components=2).fit(Xs); R=real2.transform(Xs); G=real2.transform(gen)
plt.figure(figsize=(7,5.5))
plt.scatter(R[:,0],R[:,1],s=8,c='#bbbbbb',label='真实街区',alpha=0.6)
plt.scatter(G[:,0],G[:,1],s=8,c='#C44E52',label='VAE 生成',alpha=0.5)
plt.title('VAE 生成的新街区(红) 盖在真实(灰)上 → 有盖到才算学会')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 5 · 三兄弟总整理

- **PCA**：最快最简单，只会直线投影。先用它看数据长相。
- **AE**：会弯曲，压缩/聚类通常更好；但不能生成。
- **VAE**：latent 整齐 → **能生成新数据**，是 GAN/Diffusion 之前最该懂的生成模型基石。

选择：**只想看 → PCA**；**想压得更好 → AE**；**想造新数据 → VAE**。

概念懂了，回去把隔壁 **`1_PCA` → `2_AE` → `3_VAE`** 三本各跑一遍，
每一进程式你在第 1、2 本都学过了。